# XAI for Medical Diagnosis — Demo Notebook

This notebook walks through the full pipeline:
1. Data exploration
2. Preprocessing
3. Model training & evaluation
4. Single-patient prediction
5. SHAP explanations (summary + waterfall)
6. Batch prediction

## 0. Setup

In [ ]:
# Install dependencies if needed
# !pip install pandas numpy scikit-learn shap matplotlib

import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Add src/ to the path so we can import our modules directly
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
SRC_PATH  = os.path.join(REPO_ROOT, 'src')
sys.path.insert(0, SRC_PATH)

DATA_PATH    = os.path.join(REPO_ROOT, 'data', 'sample_patient_data.csv')
RESULTS_PATH = os.path.join(REPO_ROOT, 'results')

print('Repo root:', REPO_ROOT)
print('Data file:', DATA_PATH)

## 1. Data Exploration

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
df.head(10)

In [ ]:
df.describe()

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())

print('\nStatus distribution:')
print(df['status'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
features = ['heart_rate', 'blood_pressure', 'spo2', 'temperature']

for ax, feat in zip(axes, features):
    for status, grp in df.groupby('status'):
        grp[feat].hist(ax=ax, alpha=0.6, label=status, bins=15)
    ax.set_title(feat)
    ax.legend(fontsize=8)

plt.suptitle('Feature Distributions by Patient Status', y=1.02)
plt.tight_layout()
plt.show()

## 2. Preprocessing

In [ ]:
from preprocessing import preprocess

data = preprocess(DATA_PATH)

print('Training set size :', len(data['X_train']))
print('Test set size     :', len(data['X_test']))
print('Feature columns   :', data['feature_names'])
data['X_train'].head()

## 3. Model Training & Evaluation

In [ ]:
from model import train_and_evaluate

results = train_and_evaluate(data)
print('Models trained successfully.')

In [ ]:
diag_metrics   = results['diag_metrics']
status_metrics = results['status_metrics']

print('=== DIAGNOSIS MODEL ===')
print(f"Accuracy : {diag_metrics['accuracy']:.4f}")
print(diag_metrics['report'])

print('=== STATUS MODEL ===')
print(f"Accuracy : {status_metrics['accuracy']:.4f}")
print(status_metrics['report'])

# Cross-validation results
dcv = results['diag_cv']
scv = results['status_cv']
print('=== CROSS-VALIDATION (5-fold) ===')
print(f"Diagnosis  CV: {dcv['mean']:.4f} +/- {dcv['std']:.4f}  {[round(s,3) for s in dcv['cv_scores']]}")
print(f"Status     CV: {scv['mean']:.4f} +/- {scv['std']:.4f}  {[round(s,3) for s in scv['cv_scores']]}")


In [ ]:
# Save metrics to results/metrics.txt
import os
os.makedirs(RESULTS_PATH, exist_ok=True)

with open(os.path.join(RESULTS_PATH, 'metrics.txt'), 'w') as f:
    f.write('=== DIAGNOSIS MODEL ===\n')
    f.write(f"Accuracy: {diag_metrics['accuracy']:.4f}\n")
    f.write(diag_metrics['report'])
    f.write('\n=== STATUS MODEL ===\n')
    f.write(f"Accuracy: {status_metrics['accuracy']:.4f}\n")
    f.write(status_metrics['report'])

print('Metrics saved to results/metrics.txt')

## 4. Single-Patient Prediction

In [ ]:
from prediction import predict_single, format_prediction_report

sample_patient = {
    'heart_rate':     110,
    'blood_pressure': 155,
    'spo2':           91,
    'temperature':    38.4,
}

prediction = predict_single(
    results['diag_model'],
    results['status_model'],
    data['scaler'],
    sample_patient,
)

print(format_prediction_report(prediction))

## 5. SHAP Explanations

In [ ]:
import shap
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend for saving plots
import matplotlib.pyplot as plt

from explainability import (
    compute_shap_values,
    plot_summary,
    plot_waterfall,
    explain_single,
)

In [ ]:
# Compute SHAP values for the diagnosis model on the test set
shap_vals_diag = compute_shap_values(results['diag_model'], data['X_test'])
print('SHAP values computed. Shape:', shap_vals_diag.values.shape)

In [ ]:
# --- Summary plot (global feature importance) ---
summary_path = os.path.join(RESULTS_PATH, 'shap_plots.png')
plot_summary(
    shap_vals_diag,
    data['X_test'],
    title='SHAP Summary — Diagnosis Model (Positive class)',
    save_path=summary_path,
    class_index=1,
)
print('Summary plot saved.')

# Display inline
from IPython.display import Image
Image(summary_path)

In [ ]:
# --- Waterfall plot for a single test patient ---
waterfall_path = os.path.join(RESULTS_PATH, 'shap_waterfall.png')
plot_waterfall(
    shap_vals_diag,
    sample_index=0,
    title='SHAP Waterfall — Test Patient #0',
    save_path=waterfall_path,
    class_index=1,
)
print('Waterfall plot saved.')
Image(waterfall_path)

In [ ]:
# --- Text explanation for the sample patient ---
import pandas as pd

sample_df = pd.DataFrame([sample_patient])
sample_scaled = pd.DataFrame(
    data['scaler'].transform(sample_df),
    columns=data['feature_names'],
)

text_exp = explain_single(
    results['diag_model'],
    sample_scaled,
    data['feature_names'],
)
print(text_exp)

## 6. Batch Prediction

In [ ]:
from prediction import predict_batch

# Run batch inference on the raw test features
# Re-load original (unscaled) test rows for display
raw_df = pd.read_csv(DATA_PATH)
test_indices = data['X_test'].index if hasattr(data['X_test'], 'index') else range(len(data['X_test']))

batch_results = predict_batch(
    results['diag_model'],
    results['status_model'],
    data['scaler'],
    data['X_test'],
)

print('Batch predictions (first 10 rows):')
batch_results[['heart_rate', 'blood_pressure', 'spo2', 'temperature',
               'predicted_diagnosis', 'predicted_status']].head(10)

---
## Summary

| Step | Module | Output |
|------|--------|--------|
| Data loading & cleaning | `preprocessing.py` | Scaled train/test splits |
| Model training | `model.py` | Two RandomForest classifiers |
| Inference | `prediction.py` | Diagnosis + status + probabilities |
| Explainability | `explainability.py` | SHAP summary + waterfall plots |

The SHAP plots are saved in `results/` for inclusion in reports or presentations.